In [1]:
import asyncio
import concurrent.futures
import functools
import pathlib
import multiprocessing
import os
import re
import time

from IPython.display import display, Markdown
import polars
from tqdm.notebook import tqdm

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_DIR / "output_dataset"

os.environ["INPUT_PATH"] = str(INPUT_PATH)

# Extract package data from PCAPs

TBD:
- **Parallel needs to be installed for this!**
- wireshark preferences:
  ```yaml
  dtls.psk: f28aa24f8cb8a7cc483cf336c0be70047f29
  tls.psk: f28aa24f8cb8a7cc483cf336c0be70047f29
  ```
  wireshark oscore_contexts:
  ```csv
  "01","64617461","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "64617461","01","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "02","646e73","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "646e73","02","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "03","70726f7879","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  "70726f7879","03","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  ```

In [2]:
list_code(EVALUATION_DIR / "extract_from_pcaps.sh")

#!/usr/bin/env bash
#
# extract_from_pcaps.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
PROCS=$(grep -c '^processor' /proc/cpuinfo)
if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi
OUTPUT_DATASET="${OUTPUT_DATASET:-${SCRIPT_DIR}/output_dataset}"


extract_from_pcap() {
    PCAP="$1"

    if ! echo "$PCAP" | grep -Eq ".*\.upstream.pcap" && ! [ -f "${PCAP%.pcap}".eth.csv ]; then
        "${SCRIPT_DIR}/extract_eth.sh" "${PCAP}" > "${PCAP%.pcap}".eth.csv
    elif [ -f "${PCAP%.pcap}".eth.csv ]; then
        echo "\"${PCAP%.pcap}.eth.csv\" exists already" >&2
    fi
    if echo "${PCAP}" | grep -Eq "\<https-" && ! echo "$PCAP" | grep -Eq ".*\.upstream.pcap" && ! [ -f "${PCAP%.pcap}".http.csv ]; then
        "${SCRIPT_DIR}/extract_http.sh" "${PCAP}" > "${PCAP%.pcap}".http.csv
    elif echo "${PCAP}" | grep -Eq -e "-schc-" && ! echo "$PCAP" | grep -Eq ".*\.upstream.pcap" && ! [ -f "${PCAP%.pcap}".coap.csv ]; then
        if echo "$PCAP" | grep -q "oscore"; then
            OSCORE="oscore"
        fi
        "${SCRIPT_DIR}/extract_schc.py" "${PCAP%.pcap}".eth.csv | \
             text2pcap -l 101 -t "%s.%f" -q - - 2>/dev/null | \
             "${SCRIPT_DIR}/extract_coap.sh" - "${OSCORE}" > "${PCAP%.pcap}".coap.csv
    elif ! echo "${PCAP}" | grep -Eq "\<https-" && ! [ -f "${PCAP%.pcap}".coap.csv ]; then
        "${SCRIPT_DIR}/extract_coap.sh" "${PCAP}" > "${PCAP%.pcap}".coap.csv
    elif [ -f "${PCAP%.pcap}".http.csv ]; then
        echo "\"${PCAP%.pcap}.http.csv\" exists already" >&2
    elif [ -f "${PCAP%.pcap}".coap.csv ]; then
        echo "\"${PCAP%.pcap}.coap.csv\" exists already" >&2
    fi
}

export -f extract_from_pcap
export SCRIPT_DIR

find "${OUTPUT_DATASET}" -name "*.wpan.pcap" -o -name "*.upstream.pcap" | \
    parallel -j"${PROCS}" extract_from_pcap

In [3]:
list_code(EVALUATION_DIR / "extract_eth.sh")

#! /bin/sh
#
# extract_eth.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ]; then
    echo "usage: $0 <pcap file>" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch eth.src eth.dst eth.type eth.payload"
echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields -e frame.number -e frame.time_epoch -e eth.src -e eth.dst -e eth.type -e data --disable-protocol ALL --enable-protocol eth -r "${PCAP}"

In [4]:
list_code(EVALUATION_DIR / "extract_schc.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.

import argparse
import binascii
import csv
import datetime
import io
import pathlib
import re
import sys
import unittest.mock


from scenarios.schc.schc import OpenSCHCLoader


SCRIPT_PATH = pathlib.Path(__file__).resolve().parent
OPENSCHC_PATH = SCRIPT_PATH / "scenarios" / "schc" / "openschc" / "src"
ETH_COL_TYPES = {
    "frame.number": int,
    "eth.src": lambda x: bytes.fromhex(x.replace(":", "")),
    "eth.dst": lambda x: bytes.fromhex(x.replace(":", "")),
    "eth.type": lambda x: int(x[2:], base=16),
    "eth.payload": bytes.fromhex,
}
ROUTE_COL_TYPES = {
    "eth": bytes.fromhex,
}


def convert_columns(row, conversion_map):
    for col in conversion_map:
        row[col] = conversion_map[col](row[col])


def get_device_table(csvpath):
    routes_path = csvpath.parent / csvpath.name.replace(".wpan.eth.csv", ".routes.txt")
    device_table = {}
    with routes_path.open() as routes_file:
        reader = csv.DictReader(
            routes_file,
            delimiter=" ",
            fieldnames=["name", "ipv6", "eth"]
        )
        for row in reader:
            convert_columns(row, ROUTE_COL_TYPES)
            device_table[row["eth"]] = row["name"]
    return device_table


def get_rules(csvpath, device_table):
    schc_path = csvpath.parent / ".." / "scenarios" / "schc"
    rule_name = re.sub(r"^(.+-schc-[dp][12]).*", r"\1", csvpath.name)
    rule_mode = re.sub(r"^.+-schc-[dp][12](-([^_]+))?_.*", r"\2", csvpath.name)

    rev_device_table = {v: k for k, v in device_table.items()}

    if rule_mode == "min-rules":
        rules_paths = {
            rev_device_table["client"]: schc_path / f"{rule_name}-rules-min.json"
        }
    elif rule_mode == "peer-based":
        rules_paths = {}
        for rules in schc_path.glob(f"{rule_name}-rules-*.json"):
            if rules.name.endswith("-min.json"):
                continue
            else:
                host = re.sub(r".*-rules-(.*)\.json", r"\1", rules.name)
                host_addr = rev_device_table[host]
                client_addr = rev_device_table["client"]
                rules_paths[
                    bytes(a ^ b for a, b in zip(host_addr, client_addr))
                ] = rules
    else:
        rules_paths = {
            rev_device_table["client"]: schc_path / f"{rule_name}-rules.json"
        }

    for rules in rules_paths.values():
        assert rules.exists(), f"{rules} does not exist"
    return rules_paths



class DevNull:
    def __enter__(self):
        self._stdout = sys.stdout
        self._stderr = sys.stderr
        sys.stdout = io.StringIO()
        sys.stderr = io.StringIO()
        return self

    def __exit__(self, *args):
        del sys.stdout
        del sys.stderr
        sys.stdout = self._stdout
        sys.stderr = self._stderr


def dump_pkt(pkt):
    chunk_size = 16
    for i in range(0, len(pkt), chunk_size):
        chunk = pkt[i:i+chunk_size]
        print(f"{i:06x}", binascii.hexlify(chunk, " ").decode())
    print(f"{len(pkt):06x}")


def read_csv(csvfile, device_table, rules):
    reader = csv.DictReader(csvfile, delimiter="\t")
    openschc_loader = OpenSCHCLoader(OPENSCHC_PATH)

    rule_manager = openschc_loader.get_rule_manager()
    for device_id, rules_file in rules.items():
        rule_manager.Add(device=device_id, file=str(rules_file))
    peer_based = len(rules) > 1
    protocol = {}
    for key in device_table:
        protocol[key] = openschc_loader.get_protocol(
            layer2=unittest.mock.MagicMock(),
            system=unittest.mock.MagicMock(),
            role="device" if device_table[key] == "client" else "core",
        )
        protocol[key].set_rulemanager(rule_manager)
    fragment_rows = {k: [] for k in device_table}
    prev_frame_num = 0
    for i, row in enumerate(reader):
        convert_columns(row, ETH_COL_TYPES)

        assert row["frame.number"] == prev_

In [5]:
list_code(EVALUATION_DIR / "extract_coap.sh")

#! /bin/sh
#
# extract_coap.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ] && [ $# -ne 2 ]; then
    echo "usage: $0 <pcap file|-> [oscore]" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch frame.protocols dtls.record.content_type"
FIELDS="${FIELDS} coap.code coap.request_first_in coap.mid coap.token coap.opt.ctype coap.opt.block_number coap.opt.block_mflag coap.opt.block_size coap.block coap.block.reassembled.in"

if echo "$PCAP" | grep -q "oscore" || [ "$2" = "oscore" ]; then
    FIELDS="${FIELDS} oscore.code oscore.opt.ctype oscore.opt.block_number oscore.opt.block_mflag oscore.opt.block_size"
fi


echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields $(for field in ${FIELDS}; do printf -- "-e %s " "${field}"; done) -r "${PCAP}"

In [6]:
!hyperfine --warmup 0 --max-runs 1 --show-output --ignore-failure "{EVALUATION_DIR}"/extract_from_pcaps.sh

Benchmark 1: ./extract_from_pcaps.sh
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d1_cbor_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d1_cbor_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d1_json_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d1_json_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d2_cbor_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data/oscore-d2_cbor_dns_message_b64.wpan.pcap" appears to have been cut short in the middle of a packet.
tshark: The file "/home/service/pivot-eval-tbd/data

# Generate Training Data

In [7]:
schema_overrides = {
    "coap": {
        "coap.token": str,
        "coap.opt.block_number": str,
        "coap.opt.block_mflag": str,
        "coap.opt.block_size": str,
        "oscore.opt.block_number": str,
        "oscore.opt.block_mflag": str,
        "oscore.opt.block_size": str,
    },
    "coap_upstream": {
        "coap.token": str,
        "coap.opt.block_number": str,
        "coap.opt.block_mflag": str,
        "coap.opt.block_size": str,
        "oscore.opt.block_number": str,
        "oscore.opt.block_mflag": str,
        "oscore.opt.block_size": str,
    },
}

In [8]:
COLUMN_ORDER = {
    "coap": [
        "client.coap.response.recv_time_epoch",
        "client.type",
        "client.dns.qry.name",
        "client.dns.qry.type",
        "client.coap.url",
        "client.coap.media_type",
        "client.coap.response.code",
        "client.coap.response.payload",
        "frame.number",
        "frame.time_epoch",
        "frame.protocols",
        "eth.src",
        "eth.dst",
        "eth.type",
        "eth.payload",
        "ipv6.prefix",
        "dtls.record.content_type",
        "coap.is_response",
        "coap.code",
        "coap.request_first_in",
        "coap.mid",
        "coap.token",
        "coap.opt.ctype",
        "coap.block",
        "coap.block.reassembled.in",
        "coap.opt.block_number",
        "coap.opt.block_mflag",
        "coap.opt.block_size",
        "oscore.code",
        "oscore.opt.ctype",
        "oscore.opt.block_number",
        "oscore.opt.block_mflag",
        "oscore.opt.block_size",
        "upstream.frame.number",
        "upstream.frame.time_epoch",
        "upstream.frame.protocols",
        "upstream.ipv6.prefix",
        "upstream.dtls.record.content_type",
        "upstream.coap.code",
        "upstream.coap.request_first_in",
        "upstream.coap.mid",
        "upstream.coap.token",
        "upstream.coap.opt.ctype",
        "upstream.coap.block",
        "upstream.coap.block.reassembled.in",
        "upstream.coap.opt.block_number",
        "upstream.coap.opt.block_mflag",
        "upstream.coap.opt.block_size",
        "upstream.oscore.code",
        "upstream.oscore.opt.ctype",
        "upstream.oscore.opt.block_number",
        "upstream.oscore.opt.block_mflag",
        "upstream.oscore.opt.block_size",
    ],
    "http": [
        "client.http.response.recv_time_epoch",
        "client.type",
        "client.dns.qry.name",
        "client.dns.qry.type",
        "client.http.url",
        "client.http.media_type",
        "client.http.response.status",
        "client.http.response.payload",
        "frame.number",
        "frame.time_epoch",
        "frame.protocols",
        "eth.src",
        "eth.dst",
        "eth.type",
        "eth.payload",
        "ipv6.prefix",
        "tls.record.content_type",
        "tcp.analysis.acks_frame",
        "tcp.analysis.duplicate_ack_frame",
        "tcp.analysis.rto_frame",
        "tcp.reassembled_in",
        "tcp.segment",
        "http2.headers.method",
        "http2.streamid",
        "http2.request_in",
        "http2.body.reassembled.in",
        "http2.headers.content_type",
        "upstream.ipv6.prefix",
    ],
}

WORKERS = multiprocessing.cpu_count() // 2


scenario_files = {}

print(f"Start {time.time()}")

for root, dirs, files in INPUT_PATH.walk():
    for file in files:
        if match := re.search(
            r".*\.((wpan|upstream)\.(coap|http|eth)\.csv|(client|proxy)\.log)",
            file,
        ):
            scenario = file.split(".")[0]
            if scenario not in scenario_files:
                scenario_files[scenario] = []
            if not (
                file.startswith("oscore")
                or file.startswith("coaps-p")
                or file.startswith("coaps-schc-p")
                or file.startswith("coap-p")
                or file.startswith("https")
            ) and (
                match[1] == "upstream"
                or match[4] == "proxy"
            ):
                display(Markdown(f"## No upstream {scenario}"))
                continue
            elif match[1] == "client.log":
                table = "client"
            elif match[1] == "proxy.log":
                table = "proxy"
            else:
                table = match[3]
            if match[2] == "upstream":
                table = f"{table}_upstream"
            scenario_files[scenario].append(
                {
                    "table": table,
                    "path": root / file,
                }
            )


def split_stream_ids(stream_id):
    if stream_id is None:
        return stream_id
    stream_id = [int(s) for s in stream_id.split(",")]
    if all(x == 0 for x in stream_id):
        return 0
    while any(x == 0 for x in stream_id):
        stream_id.remove(0)
    assert all(x == stream_id[0] for x in stream_id), stream_id
    return stream_id[0]


def merge_tables(scenario, scenario_files):
    if (INPUT_PATH / f"{scenario}.merged.csv.gz").exists() or \
       (INPUT_PATH / f"{scenario}.training.csv.gz").exists():
        return
    tables = {}
    for file in scenario_files:
        table = file["table"]
        path = file["path"]
        tables[table] = polars.read_csv(
            path,
            separator="\t",
            schema_overrides=schema_overrides.get(table)
        )
    if scenario.startswith("https"):
        assert all(table in tables for table in ["client", "eth", "http"]), (
            f"{', '.join(table for table in ["client", "eth", "http"] if table not in tables)} missing from {scenario}"
        )
        app_protocol = "http"
        wpan_column = "http2.streamid"
        client_column = "stream_id"
        code = "status"
        
        tables["http"] = tables["http"].with_columns(
            polars.col("http2.streamid")
            .map_elements(split_stream_ids, return_dtype=polars.Int64)
            .alias("http2.streamid")
        )
    else:
        assert all(table in tables for table in ["client", "eth", "coap"]), (
            f"{', '.join(table for table in ["client", "eth", "coap"] if table not in tables)} missing from {scenario}"
        )
        assert "coap_upstream" not in tables or "proxy" in tables, (
            f"proxy missing from {scenario}"
        )
        app_protocol = "coap"
        wpan_column = "coap.token"
        client_column = "token"
        code = "code"
    df = tables["eth"].join(
        tables[app_protocol][
            [
                col for col in tables[app_protocol].columns
                if col != "frame.time_epoch"
            ]
        ],
        on="frame.number",
        how="inner"
    )
    if "coap_upstream" in tables:
        df = df.with_columns(((df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        upstream_df = tables["coap_upstream"]
        upstream_df = upstream_df.with_columns(((upstream_df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        df = df.join(
            tables["proxy"],
            left_on="coap.token",
            right_on="old_token",
            how="left",
        )
        df = df.join(
            upstream_df,
            left_on=["coap.is_response", "new_token"],
            right_on=["coap.is_response", "coap.token"],
            how="left",
            suffix=".upstream",
        )
        df = df.rename({"new_token": "coap.token.upstream"}).join(
            tables["client"],
            left_on="coap.token.upstream",
            right_on="token",
        ).rename(
            lambda c: f"upstream.{c[:-9]}" if c.endswith(".upstream") else c
        ).rename(
            {
                "timestamp": "client.coap.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": "client.coap.url",
                "media_type": "client.coap.media_type",
                "response_code": "client.coap.response.code",
                "response_payload": "client.coap.response.payload",
            }
        )
    else:
        df = df.join(
            tables["client"],
            left_on=wpan_column,
            right_on=client_column,
        ).rename(
            {
                "timestamp": f"client.{app_protocol}.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": f"client.{app_protocol}.url",
                "media_type": f"client.{app_protocol}.media_type",
                f"response_{code}": f"client.{app_protocol}.response.{code}",
                "response_payload": f"client.{app_protocol}.response.payload",
            }
        )
    
    for k in list(tables):
        del tables[k]
    del tables
    assert all(c in COLUMN_ORDER[app_protocol] for c in df.columns), (
        f"{', '.join(c for c in df.columns if c not in COLUMN_ORDER[app_protocol])} of {scenario} not in column order"
    )
    df = df.select([c for c in COLUMN_ORDER[app_protocol] if c in df.columns])
    max_len = df["eth.payload"].str.len_chars().max()
    if max_len is None:
        print(f"Skipping {scenario}")
        display(df)
        return
    df.to_pandas().to_csv(
        INPUT_PATH / f"{scenario}.merged.csv.gz",
        compression={"method": "gzip", "compresslevel": 9},
    )
    df_training = df.select(
        ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
        # add upstream mid and token to ensure only duplicate frames
        # due to upstream retransmissions are deleted
        + (["upstream.coap.mid", "upstream.coap.token"] if "upstream.coap.mid" in df.columns else [])
    ).with_columns(
        **{
            "eth.payload": polars.col("eth.payload").str.pad_end(max_len, "x")
        }
    )
    del df
    # deduplicate frames that were duplicated due to upstream retransmission
    df_training.unique(keep="last", maintain_order=True).select(
        ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
    ).to_pandas().to_csv(
        INPUT_PATH / f"{scenario}.training.csv.gz",
        compression={"method": "gzip", "compresslevel": 9},
    )
    del df_training

print(f"Running on {WORKERS} workers")
with concurrent.futures.ProcessPoolExecutor(max_workers=WORKERS) as executor:
    def _merge_tables(scenario_name):
        return merge_tables(scenario_name, scenario_files[scenario_name])
    
    for _ in tqdm(
        executor.map(
            _merge_tables,
            scenario_files.keys(),
        ),
        total=len(scenario_files),
    ):
        continue

print(f"Stop {time.time()}")

Start 1750800035.8364289
Running on 12 workers


  0%|          | 0/296 [00:00<?, ?it/s]

Stop 1750801910.8248036


# Check If Files are Complete

In [9]:
for root, dirs, files in INPUT_PATH.walk():
    for file in files:
        match = re.search(r".*\.training\.csv\.gz$", file)
        if not match:
            continue
        if file.startswith("https-"):
            df = polars.read_csv(
                root / file.replace("training.csv.gz", "wpan.http.csv"),
                separator="\t",
            )
            # TLS control packages (e.g. handshake) should be excluded
            non_app_frames = set(
                df.filter(
                    (df["tls.record.content_type"] != 23) | (df["tls.record.content_type"].is_null())
                )["frame.number"].to_list()
            )
            # pure control packages (e.g. SYN/ACK) should be excluded
            non_app_frames |= set(
                df.filter(
                    df["frame.protocols"].str.ends_with(":tcp")
                )["frame.number"].to_list()
            )
            non_app_frames |= set(
                df.filter(
                    df["http2.streamid"].str.split(",").list.eval(
                        polars.element() == "0"
                    ).list.all()
                )["frame.number"].to_list()
            )
        else:
            df = polars.read_csv(
                root / file.replace("training.csv.gz", "wpan.coap.csv"),
                separator="\t",
                schema_overrides=schema_overrides["coap"],
            )
            if file.startswith("coaps"):
                # DTLS control packages (e.g. handshake) should be excluded
                non_app_frames = set(
                    df.filter(
                        df["frame.protocols"].str.ends_with(":dtls")
                    )["frame.number"].to_list()
                )
            else:
                non_app_frames = set()
            # Empty CoAP ACKs should be excluded
            non_app_frames |= set(
                df.filter(
                    df["coap.code"] == 0
                )["frame.number"].to_list()
            )
        del df
        df = polars.read_csv(root / file, separator=",")
        exp_frames = polars.DataFrame(
            {"frame.exp": [i for i in range(1, df["frame.number"].max() + 1) if i not in non_app_frames]}
        )
        if (df.is_empty() or df.shape[0] != exp_frames.shape[0]):
            display(Markdown(f"## ❌ Error in {file}"))
            display(Markdown(f"Number of Non-App frames {len(non_app_frames)}"))
            display(Markdown("### Broken table"))
            display(df)
            display(Markdown(f"### Duplicate frame numbers"))
            display(df.filter(polars.count("frame.number").over(df.columns) > 1))
            display(Markdown(f"### Expected App frames [{df.shape[0] - len(non_app_frames)}]"))
            display(exp_frames)
            display(Markdown(f"### Different frame numbers"))
            display(
                polars.DataFrame(
                    {
                        "diff": sorted(
                            set(df["frame.number"].to_list()).difference(
                                set(exp_frames["frame.exp"].to_list()),
                            ) | 
                            set(exp_frames["frame.exp"].to_list()).difference(
                                set(df["frame.number"].to_list()),
                            )
                        )
                    }
                )
            )
            display(Markdown("### Non-App frames"))
            display(df.filter(df["frame.number"].is_in(non_app_frames)))
            display(polars.DataFrame({"non_app_frames": sorted(non_app_frames)}))
            merged = file.replace("training.csv.gz", "merged.csv.gz")
            display(Markdown(f"### [{merged}]({(root / merged).relative_to(EVALUATION_DIR)})"))
            merged_df = polars.read_csv(root / merged, separator=",")
            display(merged_df)
            if file.startswith("https-"):
                wpan_http = file.replace("training.csv.gz", "wpan.http.csv")
                display(Markdown(f"### [{wpan_http}]({(root / wpan_http).relative_to(EVALUATION_DIR)})"))
                display(polars.read_csv(root / wpan_http, separator="\t"))
            else:
                wpan_coap = file.replace("training.csv.gz", "wpan.coap.csv")
                display(Markdown(f"### [{wpan_coap}]({(root / wpan_coap).relative_to(EVALUATION_DIR)})"))
                display(polars.read_csv(root / wpan_coap, separator="\t"))
            continue
        df = df.with_columns(exp_frames)
        check = df["frame.number"] != df["frame.exp"]
        if check.any():
            display(Markdown("### Different from expected"))
            stored = file.replace(".training.csv.gz", ".training_broken.csv")
            display(Markdown(f"#### [{stored}]({(root / stored).relative_to(EVALUATION_DIR)})"))
            display(df.filter(check))
            df.write_csv(root / stored)
            break
        display(Markdown(f"## ✅ {file} OK"))
        del df

## ✅ coaps-d2_cbor_dns_cbor.training.csv.gz OK

## ✅ coaps-d1_cbor_dns_cbor.training.csv.gz OK

## ✅ coaps-d1_json_dns_cbor.training.csv.gz OK

## ✅ coaps-d1_cbor_dns_message.training.csv.gz OK

## ✅ coaps-d2_cbor_dns_message.training.csv.gz OK

...

## ✅ coaps-schc-p2_json_dns_message.training.csv.gz OK

## ✅ oscore-schc-p2_json_dns_message.training.csv.gz OK

## ✅ oscore-base-schc-p2_cbor_dns_message.training.csv.gz OK

## ✅ oscore-base-schc-p2_json_dns_cbor.training.csv.gz OK

## ✅ oscore-base-schc-p2_json_dns_message.training.csv.gz OK